# Getting to Know Your Time Series

This notebook walks through the essential first steps when working with time series data:

1. **Cleaning** -- resampling to a regular frequency, filling gaps, detecting and treating outliers
2. **Exploring** -- trend tests, autocorrelation analysis
3. **Decomposing** -- classical, Fourier, and feature-based decomposition

We use the [polars-ts](https://github.com/domenicocinque/polars-ts) library on top of
[Polars](https://pola.rs/) for fast, expressive time series operations and
[hvPlot](https://hvplot.holoviz.org/) for interactive visualizations.

**Dataset**: M5 daily unit sales (subset) from the
[Nixtla public datasets bucket](https://datasets-nixtla.s3.amazonaws.com/m5_y.parquet).

In [ ]:
# Install polars-timeseries and hvplot if not already available
import importlib

if importlib.util.find_spec("polars_ts") is None:
    %pip install -q polars-timeseries[all]
if importlib.util.find_spec("hvplot") is None:
    %pip install -q hvplot

In [ ]:
try:
    import hvplot.polars  # noqa
except ImportError:
    pass  # hvplot is optional for visualization
import numpy as np
import polars as pl

from polars_ts import (
    acf,
    detect_outliers,
    fourier_decomposition,
    impute,
    mann_kendall,
    pacf,
    resample,
    seasonal_decompose_features,
    seasonal_decomposition,
    sens_slope,
    treat_outliers,
)

## 1. Load the data

We pull five representative series from the M5 competition -- three food items, one
hobbies item, and one household item -- all from the CA_1 store. If the remote file is
unavailable we fall back to a synthetic dataset so the notebook stays runnable offline.

In [ ]:
SERIES_IDS = [
    "FOODS_1_001_CA_1",
    "FOODS_1_002_CA_1",
    "FOODS_1_003_CA_1",
    "HOBBIES_1_001_CA_1",
    "HOUSEHOLD_1_001_CA_1",
]

try:
    df = (
        pl.scan_parquet("https://datasets-nixtla.s3.amazonaws.com/m5_y.parquet")
        .filter(pl.col("unique_id").is_in(SERIES_IDS))
        .collect()
    )
    print(f"Loaded {df.shape[0]:,} rows from remote parquet.")
except Exception as exc:
    print(f"Remote load failed ({exc}); generating synthetic data.")
    rng = np.random.default_rng(42)
    dates = pl.date_range(pl.date(2011, 1, 29), pl.date(2016, 6, 19), eager=True).alias("ds")
    frames = []
    for sid in SERIES_IDS:
        n = len(dates)
        trend = np.linspace(0, 5, n)
        seasonal = 3.0 * np.sin(2 * np.pi * np.arange(n) / 7)
        noise = rng.normal(0, 1.5, n)
        y = np.maximum(trend + seasonal + noise + 10, 0)
        frames.append(pl.DataFrame({"unique_id": [sid] * n, "ds": dates, "y": y}))
    df = pl.concat(frames)
    print(f"Generated {df.shape[0]:,} synthetic rows.")

## 2. Quick look at the data

Before any transformation, inspect the shape, column types, and a visual of the raw
series to build intuition about seasonality, trends, and potential anomalies.

In [ ]:
print(f"Shape: {df.shape}")
print(f"Columns & dtypes:\n{df.dtypes}")
df.head(10)

In [ ]:
df.hvplot.line(
    x="ds",
    y="y",
    by="unique_id",
    width=900,
    height=400,
    title="Raw Daily Sales by Series",
)

## 3. Resampling

Daily data can be noisy. Resampling to a **weekly** frequency smooths out day-of-week
effects and makes trends easier to see.

In [ ]:
df_weekly = resample(df, rule="1w", agg="mean")
print(f"Weekly shape: {df_weekly.shape}")

df_weekly.hvplot.line(
    x="ds",
    y="y",
    by="unique_id",
    width=900,
    height=400,
    title="Weekly Mean Sales",
)

## 4. Handling missing values

`impute` supports several strategies. We compare **forward fill**, **linear
interpolation**, and **seasonal** imputation (which repeats the value from the same
position in the previous cycle).

In [ ]:
# Introduce ~5% missing values for demonstration
rng = np.random.default_rng(0)
mask = rng.random(df.shape[0]) < 0.05
df_with_gaps = df.with_columns(pl.when(pl.lit(mask)).then(None).otherwise(pl.col("y")).alias("y"))
print(f"Null count: {df_with_gaps['y'].null_count()}")

# Forward fill
df_ffill = impute(df_with_gaps, method="forward_fill")

# Linear interpolation
df_linear = impute(df_with_gaps, method="linear")

# Seasonal imputation (weekly seasonality → season_length=7)
df_seasonal = impute(df_with_gaps, method="seasonal", season_length=7)

print("Nulls after forward_fill:", df_ffill["y"].null_count())
print("Nulls after linear:      ", df_linear["y"].null_count())
print("Nulls after seasonal:    ", df_seasonal["y"].null_count())

## 5. Outlier detection and treatment

We use a **z-score** detector (threshold = 3) to flag extreme values, then **clip** them
back to the boundary.

In [ ]:
df_outliers = detect_outliers(df, method="zscore", threshold=3.0)
outlier_count = df_outliers.filter(pl.col("is_outlier")).shape[0]
print(f"Outliers detected (z > 3): {outlier_count} / {df.shape[0]}")

df_treated = treat_outliers(df, method="zscore", replacement="clip", threshold=3.0)

# Compare original vs treated for one series
sid = SERIES_IDS[0]
comparison = (
    df.filter(pl.col("unique_id") == sid)
    .rename({"y": "original"})
    .join(
        df_treated.filter(pl.col("unique_id") == sid).select("ds", pl.col("y").alias("treated")),
        on="ds",
    )
)

comparison.hvplot.line(
    x="ds",
    y=["original", "treated"],
    width=900,
    height=350,
    title=f"Outlier Treatment (clip) -- {sid}",
)

## 6. Trend analysis

The **Mann-Kendall test** tells us whether a monotonic trend exists (p-value), and
**Sen's slope** quantifies the median rate of change per time step.

In [ ]:
trend_df = df.group_by("unique_id").agg(
    mann_kendall(pl.col("y").cast(pl.Float64)).alias("mann_kendall_p_value"),
    sens_slope(pl.col("y").cast(pl.Float64)).alias("sens_slope"),
)
trend_df

## 7. Autocorrelation analysis

**ACF** (autocorrelation function) reveals how past values correlate with future ones.
**PACF** (partial autocorrelation) removes the effect of intermediate lags, helping us
pick AR orders. We compute up to 20 lags and visualize with bar charts.

In [ ]:
acf_df = acf(df, max_lags=20)
pacf_df = pacf(df, max_lags=20)

acf_plot = acf_df.hvplot.bar(
    x="lag",
    y="acf",
    by="unique_id",
    width=900,
    height=350,
    title="ACF (20 lags)",
    alpha=0.7,
)

pacf_plot = pacf_df.hvplot.bar(
    x="lag",
    y="pacf",
    by="unique_id",
    width=900,
    height=350,
    title="PACF (20 lags)",
    alpha=0.7,
)

(acf_plot + pacf_plot).cols(1)

## 8. Classical decomposition

We apply an **additive** seasonal decomposition with `freq=7` (daily data, weekly
seasonality). The result separates each series into *trend*, *seasonal*, and *residual*
components.

In [ ]:
decomposed = seasonal_decomposition(df, freq=7, method="additive")
decomposed.head(10)

In [ ]:
# Visualize decomposition for the first series
sid = SERIES_IDS[0]
dec_one = decomposed.filter(pl.col("unique_id") == sid)

dec_one.hvplot.line(
    x="ds",
    y=["y", "trend", "seasonal", "resid"],
    width=900,
    height=400,
    subplots=True,
    shared_axes=False,
    title=f"Additive Decomposition (freq=7) -- {sid}",
).cols(1)

## 9. Fourier decomposition

Fourier decomposition models seasonality with sine/cosine pairs. By varying
`n_fourier_terms` we control how much detail the seasonal component captures --
fewer terms give a smoother seasonal curve.

In [ ]:
fourier_5 = fourier_decomposition(df, ts_freq=365, freqs=("week",), n_fourier_terms=5)
fourier_10 = fourier_decomposition(df, ts_freq=365, freqs=("week",), n_fourier_terms=10)

# Compare seasonal components for one series
sid = SERIES_IDS[0]
cmp = (
    fourier_5.filter(pl.col("unique_id") == sid)
    .select("ds", pl.col("seasonal").alias("seasonal_5"))
    .join(
        fourier_10.filter(pl.col("unique_id") == sid).select("ds", pl.col("seasonal").alias("seasonal_10")),
        on="ds",
    )
)

cmp.hvplot.line(
    x="ds",
    y=["seasonal_5", "seasonal_10"],
    width=900,
    height=350,
    title=f"Fourier Seasonal: 5 vs 10 terms -- {sid}",
)

## 10. Decomposition features

`seasonal_decompose_features` summarizes each series into a handful of strength
metrics. In **simple** mode it returns `trend_strength`, `seasonal_strength`, and
`resid_var`. In **mstl** mode we can specify multiple seasonal frequencies and get
per-frequency seasonal strength columns.

In [ ]:
# Simple mode
features_simple = seasonal_decompose_features(df, ts_freq=365, mode="simple")
print("Simple mode features:")
features_simple

In [ ]:
# MSTL mode -- weekly, monthly, and quarterly seasonality
# Requires statsforecast: pip install polars-timeseries[forecast]
try:
    features_mstl = seasonal_decompose_features(df, ts_freq=365, mode="mstl", seasonal_freqs=[7, 28, 90])
    print("MSTL mode features:")
    print(features_mstl)
except ImportError as e:
    print(f"Skipped MSTL (optional dependency): {e}")
    features_mstl = None

In [ ]:
# Visualize feature distributions
strength_cols = [c for c in features_mstl.columns if "strength" in c]
features_mstl.hvplot.bar(
    x="unique_id",
    y=strength_cols,
    width=900,
    height=400,
    rot=30,
    title="Decomposition Strength Features (MSTL)",
    alpha=0.8,
)

## Summary

In this notebook we walked through a complete exploratory workflow for time series data:

| Step | API | Purpose |
|------|-----|---------|
| Resample | `resample` | Align to a regular, coarser frequency |
| Impute | `impute` | Fill missing values (forward fill, linear, seasonal) |
| Outlier detection | `detect_outliers` | Flag anomalous points |
| Outlier treatment | `treat_outliers` | Clip or replace outliers |
| Trend test | `mann_kendall`, `sens_slope` | Test for and quantify monotonic trends |
| Autocorrelation | `acf`, `pacf` | Understand lag structure |
| Classical decomposition | `seasonal_decomposition` | Separate trend, seasonal, residual |
| Fourier decomposition | `fourier_decomposition` | Flexible frequency-domain seasonal model |
| Feature extraction | `seasonal_decompose_features` | Summarize series characteristics |

### Further reading

- Hyndman, R.J., & Athanasopoulos, G. (2021). [*Forecasting: Principles and Practice*, 3rd ed.](https://otexts.com/fpp3/)
- Loning, M. et al. (2024). [*Modern Time Series Forecasting with Python*, 2nd ed.](https://www.packtpub.com/product/modern-time-series-forecasting-with-python-second-edition/9781803246802)
- Polars documentation: <https://docs.pola.rs/>
- hvPlot documentation: <https://hvplot.holoviz.org/>
- polars-ts repository: <https://github.com/domenicocinque/polars-ts>